# Phase 2 — Preprocessing + Filtering

**Pipeline:** PDF → sentences → rolling 10-sentence windows → ClimateBERT topic filter → `passages.jsonl`

**Carbon tracked:** ClimateBERT filtering block (label: `climatebert_filter`)

In [1]:
import json
import re
from pathlib import Path
from collections import defaultdict

import pdfplumber
import nltk
import pandas as pd
from transformers import pipeline
from codecarbon import EmissionsTracker

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

## Configuration

In [2]:
ROOT      = Path('..').resolve()
RAW_DIR   = ROOT / 'data' / 'raw'
OUT_DIR   = RAW_DIR / 'extracted'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CARBON_DIR = ROOT / 'results' / 'carbon'
CARBON_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 10   # sentences per window  (Sampson et al. §3.2)
STRIDE      = 5    # overlap stride
MIN_SENTS   = 3    # drop windows shorter than this

BRANDS = [
    {"file": "hm-2025.pdf",        "brand": "H&M",       "industry": "fashion",      "role": "greenwashing"},
    {"file": "zara-2025.pdf",      "brand": "Zara",      "industry": "fashion",      "role": "greenwashing"},
    {"file": "patagonia-2025.pdf", "brand": "Patagonia", "industry": "fashion",      "role": "positive_control"},
    {"file": "loreal-2025.pdf",    "brand": "L'Oreal",   "industry": "clean_beauty", "role": "greenwashing"},
    {"file": "sephora-2024.pdf",   "brand": "Sephora",   "industry": "clean_beauty", "role": "greenwashing"},
    {"file": "lush-2025.pdf",      "brand": "Lush",      "industry": "clean_beauty", "role": "positive_control"},
]

## Step 1 — Parse PDFs to sentences

In [ ]:
# Boilerplate patterns to strip from every page before sentence tokenization.
# These are navigation/footer elements that pdfplumber extracts as body text.
_BOILERPLATE = re.compile(
    r'We Are Lush Report Summary.*?(?=\d{2,}|$)'   # Lush sidebar header
    r'|Click here for the complete report'
    r'|Registered number\s+\d{1,2}\s+\w+\s+\d{4}\s+\d{5,}'  # company reg no.
    r'|Contents\s+\d+\s+Foreword',                 # TOC line
    re.DOTALL | re.IGNORECASE,
)

# Patterns that look like OCR column garble: 3+ consecutive single-letter tokens
_GARBLE = re.compile(r'(?<!\w)([A-Za-z])\s+([A-Za-z])\s+([A-Za-z])\s+')


def clean_page_text(text: str) -> str:
    text = _BOILERPLATE.sub(' ', text)
    text = _GARBLE.sub(' ', text)
    # Fix hyphenation: "transi-\ntion" and "transi- \ntion" → "transition"
    text = re.sub(r'-\s*\n\s*', '', text)
    # Remove bare page numbers (standalone 1–3 digit tokens between spaces)
    text = re.sub(r'(?<!\w)\d{1,3}(?!\w)', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_sentences(pdf_path: Path) -> list[str]:
    """Extract all sentences from a PDF, page by page."""
    sentences = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            text = clean_page_text(text)
            if len(text) < 50:   # skip pages that are essentially empty after cleaning
                continue
            sentences.extend(sent_tokenize(text))
    return sentences


brand_sentences: dict[str, list[str]] = {}
for meta in BRANDS:
    pdf_path = RAW_DIR / meta['file']
    sents = extract_sentences(pdf_path)
    brand_sentences[meta['brand']] = sents
    print(f"{meta['brand']:12s}: {len(sents):,} sentences")

## Step 2 — Rolling 10-sentence windows (stride = 5)

Each window is a single passage string. Windows with fewer than 3 sentences are dropped (removes decorative headers/footers that ClimateBERT might otherwise retain).

In [4]:
def build_windows(
    sentences: list[str],
    window: int = WINDOW_SIZE,
    stride: int = STRIDE,
    min_sents: int = MIN_SENTS,
) -> list[str]:
    windows = []
    for i in range(0, len(sentences), stride):
        chunk = sentences[i : i + window]
        if len(chunk) < min_sents:
            continue
        windows.append(' '.join(chunk))
    return windows


all_passages_raw: list[dict] = []
for meta in BRANDS:
    windows = build_windows(brand_sentences[meta['brand']])
    for w in windows:
        all_passages_raw.append({
            "text":     w,
            "brand":    meta['brand'],
            "industry": meta['industry'],
            "role":     meta['role'],
        })
    print(f"{meta['brand']:12s}: {len(windows):,} windows")

print(f"\nTotal passages before filtering: {len(all_passages_raw):,}")

H&M         : 821 windows
Zara        : 595 windows
Patagonia   : 374 windows
L'Oreal     : 51 windows
Sephora     : 135 windows
Lush        : 92 windows

Total passages before filtering: 2,068


## Step 3 — ClimateBERT topic filter

Model: `climatebert/distilroberta-base-climate-detector`  
Retains only windows classified as climate-relevant.  
**Carbon tracked** under label `climatebert_filter`.

In [5]:
climatebert_detector = pipeline(
    "text-classification",
    model="climatebert/distilroberta-base-climate-detector",
    truncation=True,
    max_length=512,
    device=-1,   # CPU; set device=0 for GPU
)

config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cpu


In [6]:
tracker = EmissionsTracker(
    project_name="green-claims-nlp",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    log_level="error",
)
tracker.start()

texts = [p['text'] for p in all_passages_raw]
BATCH = 64
cb_results = []
for i in range(0, len(texts), BATCH):
    batch = texts[i : i + BATCH]
    cb_results.extend(climatebert_detector(batch))
    if i % 500 == 0:
        print(f"  {i:>6,}/{len(texts):,} passages classified...")

emissions = tracker.stop()
print(f"\nclimabert_filter emissions: {emissions:.4f} kg CO2e")

[codecarbon WARNING @ 21:02:07] Multiple instances of codecarbon are allowed to run at the same time.


       0/2,068 passages classified...



climabert_filter emissions: 0.0001 kg CO2e


In [7]:
# ClimateBERT detector labels vary by model version — inspect first
print("Distinct labels:", set(r['label'] for r in cb_results[:20]))

# Retain climate-relevant passages (label == 'yes' or '1')
CLIMATE_LABELS = {'yes', '1', 'climate', 'Yes', '1.0'}
passages: list[dict] = []
for passage, result in zip(all_passages_raw, cb_results):
    if result['label'] in CLIMATE_LABELS:
        passage['passage_id'] = len(passages)
        passages.append(passage)

print(f"Retained:  {len(passages):,} passages")
print(f"Dropped:   {len(all_passages_raw) - len(passages):,} passages")
print(f"Retention: {100 * len(passages) / len(all_passages_raw):.1f}%")

Distinct labels: {'yes', 'no'}
Retained:  1,279 passages
Dropped:   789 passages
Retention: 61.8%


## Step 4 — Save outputs

In [8]:
# passages.jsonl
with open(OUT_DIR / 'passages.jsonl', 'w') as f:
    for p in passages:
        f.write(json.dumps(p) + '\n')
print(f"Saved {len(passages):,} passages → {OUT_DIR / 'passages.jsonl'}")

Saved 1,279 passages → /Users/mandy.sun/green-claims-nlp/data/raw/extracted/passages.jsonl


In [9]:
# passage_counts.csv  — flag brands with >3× median count
counts = defaultdict(int)
for p in passages:
    counts[p['brand']] += 1

meta_map = {m['brand']: m for m in BRANDS}
count_rows = [
    {
        'brand':         b,
        'industry':      meta_map[b]['industry'],
        'role':          meta_map[b]['role'],
        'passage_count': c,
    }
    for b, c in counts.items()
]
count_df = pd.DataFrame(count_rows)
count_df['pct_of_total'] = (
    count_df['passage_count'] / count_df['passage_count'].sum() * 100
).round(1)
count_df.sort_values('passage_count', ascending=False, inplace=True)
count_df.to_csv(OUT_DIR / 'passage_counts.csv', index=False)

median_count = count_df['passage_count'].median()
flagged = count_df[count_df['passage_count'] > 3 * median_count]
print(count_df.to_string(index=False))
if not flagged.empty:
    print(f"\n⚠  Brands with >3× median ({median_count:.0f}) passage count: {flagged['brand'].tolist()}")
else:
    print(f"\nAll brands within 3× median ({median_count:.0f}) — corpus balance OK")

    brand     industry             role  passage_count  pct_of_total
      H&M      fashion     greenwashing            414          32.4
     Zara      fashion     greenwashing            385          30.1
Patagonia      fashion positive_control            306          23.9
  Sephora clean_beauty     greenwashing             95           7.4
     Lush clean_beauty positive_control             60           4.7
  L'Oreal clean_beauty     greenwashing             19           1.5

All brands within 3× median (200) — corpus balance OK
